# 第11回・第12回　逆振り子の安定化とニュートン法

このノートは2回分をまとめたものです。

1. 逆振り子に PD 制御を入れ、非線形の相図と線形近似の相図を並べて見る
2. スライドのニュートン法を、3問そのままプログラムにする
3. 時間が余ったときだけ、スライドの線形連立方程式を解く

`TODO` と書かれた行だけ、自分で埋めてください。それ以外はそのまま実行できます。


## 1. 逆振り子

スライドと同じ記号を使います。

- $\varphi$ は、**真上**を 0 とした角度です。真下を 0 とする演習1とは向きが違います。
- 入力 $u$ は、トルクを $ml^2$ で割ったものです。
- 制御を入れる前の運動方程式は

$$
\ddot{\varphi} = \frac{g}{l}\sin\varphi + u
$$

- PD 制御は、目標 $\varphi=0$ からのずれと、角速度でトルクを決めます。

$$
u = k_p(0-\varphi) - k_d\dot{\varphi}
$$

制御なしのときは、$u$ は外から与える数でした。PD では $u$ が $\varphi$ と $\dot{\varphi}$ で決まるので、**微分方程式の右辺そのものが変わります**。下の関数には、その右辺だけを書いてください。

状態は $z=(\varphi,\ \omega)$、$\omega=\dot{\varphi}$ です。固定点は $(\varphi,\omega)=(0,0)$ です。

線形近似は $\sin\varphi\approx\varphi$ としたもので、スライドの行列

$$
\frac{d}{dt}
\begin{bmatrix}\delta\varphi\\ \delta\omega\end{bmatrix}
=
\begin{bmatrix}0&1\\ g/l-k_p& -k_d\end{bmatrix}
\begin{bmatrix}\delta\varphi\\ \delta\omega\end{bmatrix}
$$

と同じです。この関数は穴埋めにしません。自分で書いた式と比べるためのものです。

ゲインの例（$g=9.8$、$l=1$）

- $k_p=14.8$、$k_d=2$：安定スパイラル（極 $-1\pm 2i$）
- $k_p=5$、$k_d=2$：P が足りないので、まだ鞍点


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

# スライドの数値。ゲインを変えたら、このセルを再実行する。
g = 9.8
l = 1.0
kp = 14.8
kd = 2.0


def pendulum_pd(t, z, g, l, kp, kd):
    """PD 制御を入れた逆振り子。z = [phi, omega]。"""
    phi, omega = z
    # 制御なしなら、u は定数だった。
    # PD では u が phi と omega の関数になる。右辺は次の2行。
    # 穴埋め: スライドの u = kp*(0 - phi) - kd*omega
    u = TODO
    # 穴埋め: スライドの phi'' = (g/l)*sin(phi) + u
    domega = TODO
    return [omega, domega]


def pendulum_linear(t, z, g, l, kp, kd):
    """真上の近くだけの近似。sin(phi) を phi に置き換えた式。"""
    phi, omega = z
    domega = (g / l - kp) * phi - kd * omega
    return [omega, domega]


def vector_field(func, phi_axis, omega_axis):
    """相図用に、格子の各点で (dphi/dt, domega/dt) を計算する。"""
    PHI, OMEGA = np.meshgrid(phi_axis, omega_axis)
    U = np.zeros_like(PHI)
    V = np.zeros_like(PHI)
    for i in range(PHI.shape[0]):
        for j in range(PHI.shape[1]):
            dphi, domega = func(0.0, [PHI[i, j], OMEGA[i, j]], g, l, kp, kd)
            U[i, j] = dphi
            V[i, j] = domega
    return PHI, OMEGA, U, V


def trajectories(func, ics, t_end=6.0):
    curves = []
    t = np.linspace(0.0, t_end, 400)
    for z0 in ics:
        sol = solve_ivp(lambda t, z: func(t, z, g, l, kp, kd),
                        (0.0, t_end), z0, t_eval=t, rtol=1e-6, atol=1e-8)
        curves.append(sol.y)
    return curves


phi_axis = np.linspace(-np.pi, np.pi, 45)
omega_axis = np.linspace(-8.0, 8.0, 45)
# 青: 真上の近く。赤: 真下に近い位置。左右の図で同じ初期値。
ics = [(0.4, 0.0), (2.8, 0.0)]
colors = ["C0", "C3"]

A = np.array([[0.0, 1.0], [g / l - kp, -kd]])
print("linear eigenvalues:", np.linalg.eigvals(A))

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
for ax, func, title in [
    (axes[0], pendulum_pd, "Nonlinear, with PD"),
    (axes[1], pendulum_linear, "Linear approximation"),
]:
    PHI, OMEGA, U, V = vector_field(func, phi_axis, omega_axis)
    ax.streamplot(PHI, OMEGA, U, V, color="0.75", density=1.1, arrowsize=1.1)
    for z0, color, label in zip(ics, colors, ["near", "far"]):
        curve = trajectories(func, [z0])[0]
        ax.plot(curve[0], curve[1], color=color, lw=2.0, label=label)
        ax.plot(z0[0], z0[1], "o", color=color)
    ax.plot(0.0, 0.0, "o", color="k", ms=6)
    ax.set_title(title)
    ax.set_xlabel("phi [rad]")
    ax.set_ylabel("omega [rad/s]")
    ax.set_xlim(phi_axis[0], phi_axis[-1])
    ax.set_ylim(omega_axis[0], omega_axis[-1])
    ax.legend(loc="upper right")
fig.tight_layout()
plt.show()


## 2. ニュートン法

スライドの3問です。各問のプログラムは、その問のセル1つに入っています。$f$ と導関数、ヤコビ行列は書いてあります。**更新式の行だけ**埋めてください。

手順はスライドの流れ図と同じです。

1. $\delta x$ を計算する
2. $|\delta x|$ が十分小さければ終了
3. $x \leftarrow x+\delta x$

1次では $\delta x = -f(x)/f'(x)$ です。2次では $J(x)\,\delta x = -f(x)$ を解きます。


### ① 立方根

$\sqrt[3]{13}$ を、$f(x)=x^3-13=0$ の根として求めます。初期値はスライドと同じ $x_0=2$ です。真値は約 $2.35133468772$ です。


In [ ]:
import numpy as np


def newton_1d(f, df, x0, tol=1e-10, max_iter=15):
    """1変数のニュートン法。途中の x をリストで返す。"""
    x = float(x0)
    history = [x]
    for k in range(max_iter):
        fx = f(x)
        dfx = df(x)
        # 導関数が 0 だと接線が引けない。更新式の穴埋めとは別の話。
        if dfx == 0.0:
            print("f'(x) = 0 なので、この点からは更新できません。 x =", x)
            return x, history
        # 1. δx = -f(x) / f'(x)
        # 3. x_new = x + δx
        # 穴埋め: この2つをまとめて、x_new を書いてください
        x_new = TODO
        history.append(x_new)
        # 2. 収束判定。|δx| が十分小さければ止まる。
        if abs(x_new - x) < tol:
            return x_new, history
        x = x_new
    print("最大回数まで回しても止まりませんでした。")
    return x, history


def f_cbrt(x):
    return x**3 - 13.0


def df_cbrt(x):
    return 3.0 * x**2


root, history = newton_1d(f_cbrt, df_cbrt, x0=2.0)
print("history:", [round(v, 10) for v in history])
print("root   :", root)
print("13**(1/3):", 13.0 ** (1.0 / 3.0))


### ② 2元の連立方程式

スライドと同じ式です。根は $(1,0)$ と $(0,1)$ です。

$$
f(x,y)=\begin{bmatrix}x^2+y-1\\ x+y^2-1\end{bmatrix}=0,
\qquad
J(x,y)=\begin{bmatrix}2x&1\\ 1&2y\end{bmatrix}
$$

初期値は3つです。$(0.8,\,0.2)$ と $(0.2,\,0.8)$ は1ステップでスライドの手計算と一致します。$(0.5,\,0.5)$ はヤコビ行列が特異で、更新できません。

$J\delta=-f$ は、自分で逆行列を計算せず、`np.linalg.solve(行列, 右辺)` で解きます。第1引数が左の行列、第2引数が右辺です。セルの先頭に、数字の小さい例を置いてあります。


In [ ]:
import numpy as np

# np.linalg.solve(A, b) は A @ x = b の x を返す。
# 逆行列は自分で作らない。第1引数が行列、第2引数が右辺。
# 例: [[2, 0], [0, 4]] @ x = [6, 8] なら x = [3, 2]
demo_A = np.array([[2.0, 0.0], [0.0, 4.0]])
demo_b = np.array([6.0, 8.0])
print("solve の例:", np.linalg.solve(demo_A, demo_b))


def f_2d(z):
    x, y = z
    return np.array([x**2 + y - 1.0, x + y**2 - 1.0])


def J_2d(z):
    x, y = z
    return np.array([[2.0 * x, 1.0], [1.0, 2.0 * y]])


def newton_2d(f, J, z0, tol=1e-10, max_iter=15):
    """2変数のニュートン法。z は長さ2の配列。"""
    z = np.array(z0, dtype=float)
    history = [z.copy()]
    for k in range(max_iter):
        fz = f(z)
        Jz = J(z)
        # 1. Jz @ delta = -fz を delta について解く。
        # 上の例と同じ。第1引数は Jz、第2引数は -fz。
        # 穴埋め: delta = np.linalg.solve(...)
        try:
            delta = TODO
        except np.linalg.LinAlgError:
            print("ヤコビ行列が特異です。この初期値からは更新できません。", z)
            return z, history
        # 3. 解の更新。ここは穴埋めではない。
        z_new = z + delta
        history.append(z_new.copy())
        if np.linalg.norm(delta) < tol:
            return z_new, history
        z = z_new
    print("最大回数まで回しても止まりませんでした。")
    return z, history


starts = [(0.8, 0.2), (0.2, 0.8), (0.5, 0.5)]
for z0 in starts:
    root, history = newton_2d(f_2d, J_2d, z0)
    print("start", z0)
    print("  1 step :", history[1] if len(history) > 1 else "更新なし")
    print("  root   :", root)


### ③ 届かない初期値

$f(x)=x^3-2x+2=0$ です。実根は約 $-1.769$ が1つあります。

スライドの指示どおり、まず初期値 $x=0$ で回してください。この初期値は根に近づかず、$0$ と $1$ を往復します。更新式が合っていても、初期値によっては解けません。


In [ ]:
# ①で作った newton_1d を使います。更新式の穴埋めは①だけです。


def f_cycle(x):
    return x**3 - 2.0 * x + 2.0


def df_cycle(x):
    return 3.0 * x**2 - 2.0


# スライドの指定。0 と 1 を往復する。
root0, history0 = newton_1d(f_cycle, df_cycle, x0=0.0, max_iter=8)
print("x0 = 0 :", [round(v, 6) for v in history0])

# 比べるための別の初期値。こちらは実根に寄る。穴埋めは増えていません。
root1, history1 = newton_1d(f_cycle, df_cycle, x0=-1.5)
print("x0 = -1.5 :", [round(v, 6) for v in history1])
print("root :", root1)


## 3. 線形連立方程式（余った時間用）

スライドの Gauss の消去法と同じ方程式です。手計算の代わりに、②で使った `np.linalg.solve(行列, 右辺)` で解きます。消去法のプログラムは書きません。

$$
A=
\begin{bmatrix}
2&2&3\\
1&1&2\\
1&0&3
\end{bmatrix},
\qquad
b=
\begin{bmatrix}1\\ 2\\ 3\end{bmatrix}
$$


In [ ]:
import numpy as np

A = np.array([
    [2.0, 2.0, 3.0],
    [1.0, 1.0, 2.0],
    [1.0, 0.0, 3.0],
])
b = np.array([1.0, 2.0, 3.0])

# 穴埋め: Ax = b の解。np.linalg.solve(A, b)
x = TODO
print("x =", x)
print("A @ x =", A @ x)  # b と一致すれば、解けている
